# SPARQLing FAIRyland — map-ready SPARQL for QGIS

*A SPARQLing Unicorn demonstration notebook (local Jupyter variant).*

## About this notebook

Every query here follows the **SPARQLing Unicorn convention**: it binds **`?item`** (the
main feature) and **`?geo`** (its `geo:asWKT` geometry), so each result **renders as a map
here** and can be **pasted straight into the plugin’s query editor**
(*Linked Data Processing → Concept Query*) to load as a QGIS vector layer. Optional
`?itemLabel` (the find’s name) and `?layerLabel` (its class) come along as attributes you
can style by in QGIS.

The dataset is *FAIRyland — “The Lost Flat-Pack Civilisation of Norrfors”*, a wholly
fictional GeoSPARQL excavation dataset (finds, trenches and periods are invented; only the
geological dates are real).

**What you’ll learn.** Four reusable, map-producing SPARQL patterns:

1. **all finds** with geometry, styled by class (`?layerLabel`);
2. the **trench plan** — the 30 trenches as polygons;
3. finds of a chosen **geological period**, via property-path roll-up (`hasperiod/skos:broader*`);
4. finds **within a chosen trench** (the GeoSPARQL `geof:sfWithin` form is noted for GeoSPARQL endpoints).

**Data-context notes.** Coordinates use **CRS84** (lon, lat; WGS 84); geometries are points
and polygons (finds, trenches) plus lines (baulk paths).

**Tooling notes.** We query the Turtle file locally with **`rdflib`** — exactly what the
SPARQLing Unicorn plugin does on a local `.ttl`. The `?item` + `?geo` binding is all the
plugin needs to turn each row into a feature.

> A browser-executable **`.qmd`** companion (Pyodide, no install) will accompany this in the OER.

### Requirements

```bash
pip install rdflib shapely folium pandas
```


In [ ]:
# --- Setup: imports, data source, graph, and a reusable map helper ---
import pandas as pd
from rdflib import Graph
from shapely import wkt as shapely_wkt
import folium

TTL_SOURCE = "fairyland.ttl"
TTL_URL    = "https://raw.githubusercontent.com/unicorn-demo/test1/main/fairyland.ttl"

g = Graph()
try:
    g.parse(TTL_SOURCE, format="turtle")
except Exception:
    g.parse(TTL_URL, format="turtle")

PFX = """PREFIX fairyland: <https://github.com/Research-Squirrel-Engineers/FAIRyland/>
PREFIX geo:  <http://www.opengis.net/ont/geosparql#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX suni: <http://www.github.com/sparqlunicorn#>"""

PALETTE = ["#8e44ad", "#2980b9", "#f39c12", "#c0392b", "#16a085", "#7f8c8d",
           "#27ae60", "#34495e", "#111111", "#e84393", "#00897b", "#d35400"]

def map_geo(query, colour_by=None, zoom=17, weight=2):
    """Run a SPARQL SELECT that binds ?item and ?geo (a WKT literal) and map every
    geometry. Points -> circle markers; polygons/lines -> outlines. If `colour_by` names
    another result column (e.g. 'layerLabel'), features are coloured by it with a legend."""
    res  = g.query(query)
    vrs  = [str(v) for v in res.vars]
    assert "geo" in vrs, "the query must bind a ?geo (WKT) column"
    rows = [{str(c): (None if r[c] is None else str(r[c])) for c in res.vars} for r in res]
    geoms = [(row, shapely_wkt.loads(row["geo"])) for row in rows]
    cs = [gm.centroid for _, gm in geoms]
    m = folium.Map(location=[sum(p.y for p in cs)/len(cs), sum(p.x for p in cs)/len(cs)],
                   zoom_start=zoom, tiles="OpenStreetMap")
    cmap = {}
    if colour_by:
        vals = sorted({row.get(colour_by) for row in rows if row.get(colour_by)})
        cmap = {val: PALETTE[i % len(PALETTE)] for i, val in enumerate(vals)}
    for row, gm in geoms:
        col = cmap.get(row.get(colour_by), "#8e44ad") if colour_by else "#8e44ad"
        popup = folium.Popup(" | ".join(f"{k}={row[k]}" for k in vrs if k != "geo"), max_width=260)
        if gm.geom_type in ("Polygon", "MultiPolygon"):
            for poly in ([gm] if gm.geom_type == "Polygon" else list(gm.geoms)):
                folium.Polygon([(y, x) for x, y in poly.exterior.coords], color=col,
                               weight=weight, fill=True, fill_opacity=0.15, popup=popup).add_to(m)
        elif gm.geom_type == "LineString":
            folium.PolyLine([(y, x) for x, y in gm.coords], color=col, weight=3).add_to(m)
        else:
            folium.CircleMarker([gm.centroid.y, gm.centroid.x], radius=4, color=col,
                                fill=True, fill_opacity=0.85, popup=popup).add_to(m)
    if cmap:
        legend = ("<div style='position:fixed;bottom:20px;left:20px;z-index:9999;background:white;"
                  "padding:8px 10px;border:1px solid #999;border-radius:4px;font:12px sans-serif'>"
                  f"<b>{colour_by}</b><br>"
                  + "".join(f"<span style='color:{c}'>●</span> {v}<br>" for v, c in cmap.items())
                  + "</div>")
        m.get_root().html.add_child(folium.Element(legend))
    return m

print(f"✓ Graph loaded: {len(g):,} triples. Every query below binds ?item + ?geo and maps.")

## 1 — All finds, styled by class

The overview layer. `?item` is the find, `?geo` its WKT geometry, `?layerLabel` its class
(for category styling in QGIS), `?itemLabel` its name.

In [ ]:
q_all = PFX + """
SELECT ?item ?itemLabel ?geo ?layerLabel WHERE {
  ?item a ?cls .
  ?cls  rdfs:subClassOf fairyland:Find ; rdfs:label ?layerLabel .
  OPTIONAL { ?item suni:Name ?itemLabel }
  ?item geo:hasGeometry/geo:asWKT ?geo .
}"""
print(q_all)
map_geo(q_all, colour_by="layerLabel")

## 2 — The trench plan

The 30 trenches as **polygons** — a pure-geometry layer (`?item` = trench, `?geo` = its
outline), ideal as a base "site plan" over a satellite basemap in QGIS.

In [ ]:
q_trenches = PFX + """
SELECT ?item ?itemLabel ?geo WHERE {
  ?item a fairyland:Trench ;
        rdfs:label ?itemLabel ;
        geo:hasGeometry/geo:asWKT ?geo .
}"""
print(q_trenches)
map_geo(q_trenches, zoom=16, weight=1)

## 3 — Finds of a chosen period

Select finds by **geological period**, rolling sub-phases up to their umbrella with
`hasperiod/skos:broader*`. Change `fairyland:Cretaceous` to `Triassic`, `Jurassic`, or
`Mesozoic` (the umbrella returns everything).

In [ ]:
PERIOD = "Cretaceous"   # Triassic | Jurassic | Cretaceous | Mesozoic
q_period = PFX + f"""
SELECT ?item ?itemLabel ?geo ?layerLabel WHERE {{
  ?item fairyland:hasperiod/skos:broader* fairyland:{PERIOD} ;
        a ?cls .
  ?cls  rdfs:subClassOf fairyland:Find ; rdfs:label ?layerLabel .
  OPTIONAL {{ ?item suni:Name ?itemLabel }}
  ?item geo:hasGeometry/geo:asWKT ?geo .
}}"""
print(q_period)
map_geo(q_period, colour_by="layerLabel")

## 4 — Finds within a chosen trench

Finds recorded in one trench, returned with geometry. Change `fairyland:Trench_C3` to any
cell `A1…E6`. On a **GeoSPARQL-aware** endpoint the *spatial* form selects whatever falls
inside the trench polygon directly:

```sparql
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
SELECT ?item ?geo WHERE {
  fairyland:Trench_C3 geo:hasGeometry/geo:asWKT ?tWkt .
  ?item a ?c . ?c rdfs:subClassOf fairyland:Find .
  ?item geo:hasGeometry/geo:asWKT ?geo .
  FILTER(geof:sfWithin(?geo, ?tWkt))
}
```

Both bind `?item` + `?geo` and map identically for this dataset.

In [ ]:
TRENCH = "Trench_C3"   # A1 ... E6
q_in_trench = PFX + f"""
SELECT ?item ?itemLabel ?geo ?layerLabel WHERE {{
  ?item fairyland:hastrench fairyland:{TRENCH} ;
        a ?cls .
  ?cls  rdfs:subClassOf fairyland:Find ; rdfs:label ?layerLabel .
  OPTIONAL {{ ?item suni:Name ?itemLabel }}
  ?item geo:hasGeometry/geo:asWKT ?geo .
}}"""
print(q_in_trench)
map_geo(q_in_trench, colour_by="layerLabel", zoom=19)

## Explore

Every map above came from one `SELECT … ?item … ?geo`. Combine the patterns — e.g.
*Cretaceous bananas only* — and the result still binds `?item` + `?geo`, so it maps here
and loads as a QGIS layer in the Unicorn plugin. Edit and re-run:

In [ ]:
q_custom = PFX + """
SELECT ?item ?geo WHERE {
  ?item a fairyland:Banana ;
        fairyland:hasperiod/skos:broader* fairyland:Cretaceous ;
        geo:hasGeometry/geo:asWKT ?geo .
}"""
print(q_custom)
map_geo(q_custom)

---

*Part of the NFDI4Objects Open Educational Resources. Dataset: FAIRyland (fictional),
Research Squirrel Engineers Network, CC BY 4.0. Local Jupyter variant; a browser-executable
Quarto Live (`.qmd`) companion accompanies it in the OER.*